# Random Forest Ensemble to Predict Diabetes
- Titled "diabetes_prediction_dataset.csv"
- 100,000 entries

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
data_filepath = '../data/external/diabetes_prediction_dataset.csv'
Kaggle_Diabetes_Data = pd.read_csv(data_filepath)

## Data Preprocessing

**Summary of Dataset**

In [ ]:
print("This is the Data distribution of all variables in the dataset")
print(Kaggle_Diabetes_Data.describe()) # Data distribution of variables
print("\nThese are the number of columns in the dataset")
print(Kaggle_Diabetes_Data.columns)
print("\nThese are the number of people with and without diabetes")
print(Kaggle_Diabetes_Data['diabetes'].value_counts())

**Data distribution of both numerical and categorical columns**

In [ ]:
# Data distribution of categorical columns
categorical_columns = Kaggle_Diabetes_Data.select_dtypes(include=['object']).columns
for col in categorical_columns:
    print(f"This is the data distribution of categorical column `{col}`:")
    print(Kaggle_Diabetes_Data[col].value_counts())
    print("-"*50)


In [ ]:
# Checking for missing values
print("Missing values in Dataset")
Kaggle_Diabetes_Data.isnull().sum()

In [ ]:
# Histograms for Numerical Variables (individually)
sns.set(style="whitegrid")
numerical_columns = Kaggle_Diabetes_Data.select_dtypes(include=['number']).columns

for col in numerical_columns:
    plt.figure(figsize=(8,4))
    sns.histplot(Kaggle_Diabetes_Data[col], kde='True')
    plt.title(f"Histogram of {col}")
plt.show()

In [ ]:
# Visualizing BMI Distributions as max value of 95.69 suggest the presence of outliers or extreme values
plt.figure(figsize=(6, 4))
sns.histplot(Kaggle_Diabetes_Data['bmi'], kde=True)
plt.title("Histogram of BMI")
plt.show()

# Box Plot
plt.figure(figsize=(8, 6))
sns.boxplot(x=Kaggle_Diabetes_Data['bmi'])
plt.title("Boxplot of BMI")
plt.show()

# Calculating Inter Quartile Range for Outlier Detection
Q1 = Kaggle_Diabetes_Data['bmi'].quantile(0.25)
Q3 = Kaggle_Diabetes_Data['bmi'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
print("This is the lower bound being used for outlier detection:", lower_bound)
print("This is the upper bound being used for outlier detection:", upper_bound)

outliers = Kaggle_Diabetes_Data[(Kaggle_Diabetes_Data['bmi'] < lower_bound) | (Kaggle_Diabetes_Data['bmi'] > upper_bound)]
print("Number of Outliers Detected in BMI:", outliers.shape[0])

In [ ]:
# Handling the outliers
# Dropping participants with BMI outside the lower and upper bounds
Kaggle_Diabetes_Data = Kaggle_Diabetes_Data[(Kaggle_Diabetes_Data['bmi'] >= lower_bound) & (Kaggle_Diabetes_Data['bmi'] <= upper_bound)]

# Print the shape of the dataset to confirm the number of rows after removal
print("Number of entries after removing BMI outliers:", Kaggle_Diabetes_Data.shape[0])

# Rechecking Data Distribution of BMI
print(Kaggle_Diabetes_Data.describe())
print("\nThese are the number of people with and without diabetes")
print(Kaggle_Diabetes_Data['diabetes'].value_counts())

In [ ]:
# Bar charts of Categorical columns
sns.set(style="whitegrid")
for col in categorical_columns:
    plt.figure(figsize=(8, 4))
    sns.countplot(x=col, data=Kaggle_Diabetes_Data)
    plt.title(f"Bar Chart of {col}")
    plt.xticks(rotation=45)  # Rotates the labels on the x-axis for better readability
    plt.show()

- Not Current = Refers to individuals who used to smoke but are currently not smoking
- Former = Individuals who used to smoke but are currently not smoking AND have been abstinent for a longer period of time than those in the "not current" category.
- Current = Indicates that the individual is currently a smoker at the time of data collection. It means that the person is actively smoking or has reported smoking recently.
- Ever = Represents individuals who have ever smoked in their lifetime, regardless of their current smoking status (e.g. Current and Former). It includes individuals who are currently smoking ("Current"), as well as those who have previously smoked but may have quit at the time of data collection ("Former").

**Ordering them from least to most harmful for diabetes**
1. Never
2. Not Current
3. Former
4. Current
5. Ever (dropping this column for less confusing analysis)

In [ ]:
# Correlation Matrix (features (only numerical) and target variable (diabetes))
corr_matrix = Kaggle_Diabetes_Data[numerical_columns].corr()
plt.figure(figsize=(10,8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix of Numerical Features and Diabetes')
plt.show()

**One-Hot Encoding Categorical Variables**
- No numerical relationship between categories so we just use One-Hot Encoding

In [ ]:
# Dropping all participants with the "Ever" Category for less confusing analysis
Kaggle_Diabetes_Data = Kaggle_Diabetes_Data[Kaggle_Diabetes_Data['smoking_history'] != 'ever']
print(Kaggle_Diabetes_Data['smoking_history'].value_counts())

In [ ]:
# Applying One-Hot Encoding to categorical variables
Kaggle_Diabetes_Data_encoded = pd.get_dummies(Kaggle_Diabetes_Data, columns=['gender', 'smoking_history']) # Don't use drop_first = True for non-linear models
print(Kaggle_Diabetes_Data_encoded.head())
export_filepath = "../data/processed/cleaned_diabetes_one_hot_encoding.csv"
Kaggle_Diabetes_Data_encoded.to_csv(export_filepath, index=False)

**Model Building (Initial)**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score

data_filepath = '../data/processed/cleaned_diabetes_one_hot_encoding.csv'
Kaggle_Diabetes_Data_encoded = pd.read_csv(data_filepath)


gk = Kaggle_Diabetes_Data_encoded.groupby('diabetes')
diabetes_true = gk.get_group(1)
diabetes_false = gk.get_group(0)
diabetes_false = diabetes_false.sample(8028)
Kaggle_Diabetes_Data_encoded = pd.concat([diabetes_false, diabetes_true])

# Seperating Features (X) AND target Variable (Y)
X = Kaggle_Diabetes_Data_encoded.drop('diabetes', axis=1)
Y = Kaggle_Diabetes_Data_encoded['diabetes']

# Splitting the data into training and testing sets
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [ ]:
# Training the Initial Random Forest Model
rf_model = RandomForestClassifier(
    random_state=42,
    n_estimators=100
)
rf_model.fit(X_train, Y_train)

In [ ]:
# Predictiing and Evaluating the Model
Y_pred = rf_model.predict(X_test)
print("Accuracy:", accuracy_score(Y_test, Y_pred))
print("ROC AUC:", roc_auc_score(Y_test, Y_pred))
print("Confusion Matrix:\n", confusion_matrix(Y_test, Y_pred))
print("Classification Report:\n", classification_report(Y_test, Y_pred, digits=15))

**Feature Engineering before Balancing the Dataset**

In [ ]:
# Adding interaction terms
# We add these interaction terms to capture the combined effects of these variables
# For example, the risk of diabetes might be higher in individuals who have both high BMI and are older.
Kaggle_Diabetes_Data['age_bmi_interaction'] = Kaggle_Diabetes_Data['age'] * Kaggle_Diabetes_Data['bmi']
Kaggle_Diabetes_Data['hypertension_heart_interaction'] = Kaggle_Diabetes_Data['hypertension'] * Kaggle_Diabetes_Data['heart_disease']

# Applying Transformations
Kaggle_Diabetes_Data['log_bmi'] = np.log(Kaggle_Diabetes_Data['bmi'] + 1) # Adding 1 to avoid log(0)
Kaggle_Diabetes_Data['sqrt_age'] = np.sqrt(Kaggle_Diabetes_Data['age'])

# Re-apply one-hot encoding to include the new features
Kaggle_Diabetes_Data_encoded = pd.get_dummies(Kaggle_Diabetes_Data, columns=['gender', 'smoking_history'])

# Exporting the updated dataset with feature engineering
#export_filepath = "../data/processed/cleaned_diabetes_with_feature_engineering.csv"
#Kaggle_Diabetes_Data_encoded.to_csv(export_filepath, index=False)

data_filepath = '../data/processed/cleaned_diabetes_with_feature_engineering.csv'
Kaggle_Diabetes_Data_encoded = pd.read_csv(data_filepath)

# Update the features (X) and target (Y) for model building
X = Kaggle_Diabetes_Data_encoded.drop('diabetes', axis=1)
Y = Kaggle_Diabetes_Data_encoded['diabetes']

# Splitting the data into training and testing sets (same as before)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# Training the updated Random Forest Model
rf_model.fit(X_train, Y_train)

# Predicting and Evaluating the updated model
Y_pred = rf_model.predict(X_test)
print("Updated Model Accuracy:", accuracy_score(Y_test, Y_pred))
print("Updated Model ROC AUC:", roc_auc_score(Y_test, Y_pred))
print("Updated Model Confusion Matrix:\n", confusion_matrix(Y_test, Y_pred))
print("Updated Model Classification Report:\n", classification_report(Y_test, Y_pred))

**Balancing the Dataset**

In [ ]:
!pip install -U imbalanced-learn

In [ ]:
from imblearn.over_sampling import SMOTE

# Applying SMOTE for balancing the dataset
smote = SMOTE(random_state=42)
X_train_resampled, Y_train_resampled = smote.fit_resample(X_train, Y_train)

# Re-training the Model with Balanced Data
rf_model.fit(X_train_resampled, Y_train_resampled)

# Re-evaluating the Model
Y_pred_resampled = rf_model.predict(X_test)
print("Model Accuracy after Balancing:", accuracy_score(Y_test, Y_pred_resampled))
print("Model ROC AUC after Balancing:", roc_auc_score(Y_test, Y_pred_resampled))
cm = confusion_matrix(Y_test, Y_pred_resampled)
print("Confusion Matrix after Balancing:\n", cm)
report = classification_report(Y_test, Y_pred_resampled)
print("Classification Report after Balancing:\n", report)


**Hyperparameter Tuning**

In [ ]:
from sklearn.model_selection import GridSearchCV

# Define a wider range of hyperparameters
param_grid = {
    'n_estimators': range(50, 501, 50),
    'max_depth': list(range(5, 31, 5)) + [None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt']
}

grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, scoring='roc_auc', n_jobs=1)
grid_search.fit(X_train_resampled, Y_train_resampled)
best_params = grid_search.best_params_
best_score = grid_search.best_score_
print("Best Parameters:", best_params)
print("Best ROC AUC Score:", best_score)

**Retraining Model with best Parameters**

In [ ]:
best_rf_model = RandomForestClassifier(
    n_estimators=best_params['n_estimators'],
    max_depth=best_params['max_depth'],
    min_samples_split=best_params['min_samples_split'],
    min_samples_leaf=best_params['min_samples_leaf'],
    max_features=best_params['max_features'],
    random_state=42
)
best_rf_model.fit(X_train_resampled, Y_train_resampled)

In [ ]:
Y_pred = best_rf_model.predict(X_test)
print("Accuracy:", accuracy_score(Y_test, Y_pred))
print("ROC AUC:", roc_auc_score(Y_test, Y_pred))
print("Confusion Matrix:\n", confusion_matrix(Y_test, Y_pred))
print("Classification Report:\n", classification_report(Y_test, Y_pred))